
## Trial-wise pattern similarity between Viewing 1 and Viewing 2

For each subject and each Schaefer-100 parcel (ROI), we correlate the voxel pattern for a
given trial at Viewing 1 with the pattern for the same trial at Viewing 2. Produces the `R_Value` column used by`002._subject_by_node_Viewing1_Viewing2.Rmd`. (It will be Fisher z-transformed there)

Inputs (single-trial betas from AFNI + parcel-wise arrays) are **not** distributed
with this repo — raw MRI will be made available on OpenNeuro. The dataframe created in `data/` additionally carries behavioral, NLP columns that are merged in separately in the Fig. 2 folder (reinterpretation!)

In [1]:
import numpy as np
import pandas as pd
import tqdm
import re
from scipy.stats import pearsonr

In [2]:
save_data = '../../../data/fmri_processed/'


In [3]:
sub_list = ['sub-005', 'sub-006','sub-007', 'sub-008', 'sub-009', 'sub-010', 'sub-011','sub-012', 'sub-013', 'sub-014', 'sub-015', 'sub-016', 'sub-017', 'sub-018', 'sub-019', 'sub-020', 'sub-021', 'sub-022', 'sub-024', 'sub-025', 'sub-026', 'sub-027', 'sub-028', 'sub-029', 'sub-030', 'sub-031', 'sub-032', 'sub-033', 'sub-034', 'sub-035', 'sub-036', 'sub-037', 'sub-038', 'sub-039', 'sub-040', 'sub-041', 'sub-042', 'sub-043',  'sub-045', 'sub-046', 'sub-047', 'sub-048', 'sub-049', 'sub-050', 'sub-051', 'sub-052', 'sub-054', 'sub-056', 'sub-057', 'sub-058', 'sub-059', 'sub-060', 'sub-061', 'sub-062', 'sub-063', 'sub-064']

sub = sub_list[0]
ses = '1'
glm_name = 'trialLevelAnalyses_byImage'
task = 'madlibs'

save_data = '../../../MadLibs_pub/data/fmri_processed/' 

df = pd.read_csv(save_data + f'beta_labels/{glm_name}/{sub}_ses-0{ses}_beta_map_info.csv')
print(df.shape)
df.head(4)



(91, 6)


,beta_map_index,beta_label,trial_no,trial_type,viewing,stim_file
0,0,fstat,0,0,0,0
1,1,img_pres#0,1,1,1,/static/images/EE_15_reshaped.jpeg
2,2,img_pres#1,1,1,2,/static/images/EE_15_reshaped.jpeg
3,3,img_pres#2,2,1,1,/static/images/CDK_4_reshaped_2.5.jpeg


In [4]:
def get_indices_of_presentation(df, column, value):
    
    df = df[df[column]==value]
    
    return df, df.index.tolist()

def is_constant_array(arr):
    return np.all(arr == arr[0])

def compute_multivariate_rho(mat1, mat2):
    r = []
    for t in range(mat1.shape[1]):
        if is_constant_array(mat1[:, t]) or is_constant_array(mat2[:, t]):
            r.append(np.nan)  # or another placeholder for undefined correlation
        else:
            r.append(pearsonr(mat1[:, t], mat2[:, t])[0])
    return r



In [7]:
parcellation = '100'
node_range = range(1,101) #it is 1 in this case of the SRM-ed data! 
dir_local = 'schaefer_rois'
file_end = '100_7N'


list_of_df = []

for sub in tqdm.tqdm(sub_list): 
    try:

        df = pd.read_csv(save_data + f'beta_labels/{glm_name}/{sub}_ses-0{ses}_beta_map_info.csv')
#         node_array = np.load(f'../../../MadLibs_pub_REML/data/fmri_processed/schaefer_rois_updated_032026/{sub}_{task}_{file_end}.npy',allow_pickle=True)
        node_array = np.load(save_data + f'/{dir_local}/' + f'{sub}_{task}_{file_end}.npy',allow_pickle=True) 


        dict_r = {}

        r_cont_vals = []
        r_expt_vals = []
        r_values = []
        node_vals = []
        condition_vals = []
        subject_vals = []
        stim_file_vals = []


        for node in node_range:

            viewing1,v1_idx = get_indices_of_presentation(df, 'viewing', 1)
            viewing2,v2_idx = get_indices_of_presentation(df, 'viewing', 2)
            viewing1_ctrl,v1_ctrl_idx =  get_indices_of_presentation(viewing1, 'trial_type', 2)
            viewing1_exp,v1_expt_idx =  get_indices_of_presentation(viewing1, 'trial_type', 1)
            viewing2_ctrl,v2_ctrl_idx =  get_indices_of_presentation(viewing2, 'trial_type', 2)
            viewing2_exp,v2_expt_idx =  get_indices_of_presentation(viewing2, 'trial_type', 1)

            n_vox_v1_ctrl = node_array[node-1][:,:,v1_ctrl_idx]
            n_vox_v2_ctrl = node_array[node-1][:,:,v2_ctrl_idx]


            n_vox_v1_expt = node_array[node-1][:,:,v1_expt_idx]
            n_vox_v2_expt = node_array[node-1][:,:,v2_expt_idx]
            
#             #################################################################
#             #I think this does it!
#             n_vox_v1_ctrl = node_array[node-1][:,:,[i-1 for i in v1_ctrl_idx]] #-1 was added to the index because it is sized 90 not 91 because changed to REML
#             n_vox_v2_ctrl = node_array[node-1][:,:,[i-1 for i in v2_ctrl_idx]]


#             n_vox_v1_expt = node_array[node-1][:,:,[i-1 for i in v1_expt_idx]]
#             n_vox_v2_expt = node_array[node-1][:,:,[i-1 for i in v2_expt_idx]]
#             #################################################################

            ## Drop the middle axis (axis 1)
            n_vox_v1_ctrl = np.squeeze(n_vox_v1_ctrl, axis=1)
            n_vox_v2_ctrl = np.squeeze(n_vox_v2_ctrl, axis=1)
            n_vox_v1_expt = np.squeeze(n_vox_v1_expt, axis=1)
            n_vox_v2_expt = np.squeeze(n_vox_v2_expt, axis=1)

            r_cont = compute_multivariate_rho(n_vox_v1_ctrl,n_vox_v2_ctrl)
            r_expt = compute_multivariate_rho(n_vox_v1_expt,n_vox_v2_expt)


            r_cont_vals.append(r_cont)
            r_expt_vals.append(r_expt)

            for i in r_cont:
                r_values.append(i)
                node_vals.append(node)
                condition_vals.append('cont')
                subject_vals.append(sub)
            #doing it for all of them at once
            stim_file_vals.extend(list(viewing1.loc[list(v1_ctrl_idx)]['stim_file'])) #just one row so doing just viewing1 

            for i in r_expt:
                r_values.append(i)
                node_vals.append(node)
                condition_vals.append('expt')
                subject_vals.append(sub)
            stim_file_vals.extend(list(viewing1.loc[list(v1_expt_idx)]['stim_file']))


        df_ = pd.DataFrame({
            'R_Value': r_values,
            'Node_Number': node_vals,
            'Condition': condition_vals,
            'Subject': subject_vals,
            'stim_file': stim_file_vals
        })

        list_of_df.append(df_)
    except:
        print(f'ISSUE with {sub}')



100%|███████████████████████████████████████████| 56/56 [05:11<00:00,  5.57s/it]


In [8]:
all_comb_df = pd.concat(list_of_df)
all_comb_df.to_csv(f'../../data/subject_by_node_Viewing1_Viewing2_subs_{dir_local}.csv')


### Adding in the behavioral columns

In [9]:

# Function to load and cache behavioral data just to speed things up! 
def load_behavioral_data(sub, participant_id, cache):
    if sub not in cache:
        filename = f'../../../MadLibs/data/madlibs_8_realtime/iteration_12_fMRI/reorganized_data/{sub}/madlibs_8_realtime_participant{participant_id}_run5_behavioral_metrics_total_runs.csv'
        cache[sub] = pd.read_csv(filename)
        print(sub)
    return cache[sub]

# Preload behavioral data for subjects
behavioral_data = {}
pattern = r'sub-(\d+)'
unique_subjects = all_comb_df['Subject'].unique()

for sub in unique_subjects:
    match = re.search(pattern, sub)
    if match:
        participant_id = int(match.group(1))
        load_behavioral_data(sub, participant_id, behavioral_data)

# Process all_comb_df to extract needed values
def extract_values(row):
    sub = row['Subject']
    image_path = row['stim_file']
    
    try:
        if sub in behavioral_data:
            df_behavioral = behavioral_data[sub]
            filtered_df = df_behavioral[df_behavioral['img'] == image_path]
            
            if not filtered_df.empty:
                return (
                    filtered_df['appraisal_OTHER_2_cond1'].values[0],
                    filtered_df['cosine_sim_avg_inputs_BERT_context_miniLM_L6'].values[0],
                    filtered_df['cosine_sim_avg_inputs_BERT_context_miniLM_L12'].values[0],
                    filtered_df['cosine_sim_avg_inputs_BERT_context_mpnet'].values[0],
                    filtered_df['appraisal_SELF_2_cond1'].values[0],
                    filtered_df['appraisal_SELF_1'].values[0],
                    filtered_df['appraisal_SELF_2_cond2'].values[0],
                    filtered_df['sentiment_score'].values[0],
                    filtered_df['input1'].values[0],
                    filtered_df['input2'].values[0],
                    filtered_df['input3'].values[0],
                    filtered_df['other_input1'].values[0],
                    filtered_df['other_input2'].values[0],
                    filtered_df['other_input3'].values[0],
                    filtered_df['appraisal_SELF_1_RT'].values[0]
                   
                )
        
        return (None, None, None, None, None, None, None, None, None, None, None)
        
    except Exception as e:
        #Should not generate any error! 
        print(f"ERROR with subject: {sub}, image: {image_path}")
        print(f"Error details: {e}")
        if sub in behavioral_data:
            print(f"Available columns for {sub}: {list(behavioral_data[sub].columns)}")
        raise  #

# Vectorize the operation with apply
all_comb_df[['appraisal_OTHER_cond1', 
             'cosine_sim_avg_inputs_BERT_context_miniLM_L6', 
             'cosine_sim_avg_inputs_BERT_context_miniLM_L12', 
             'cosine_sim_avg_inputs_BERT_context_mpnet',
             'appraisal_SELF_cond1', 
             'confidence', 
             'appraisal_SELF_cond2', 
             'sentiment_score',
             'input1',
             'input2',
             'input3',
             'other_input1',
             'other_input2',
             'other_input3',
             'appraisal_SELF_1_RT',
            ]] = all_comb_df.apply(extract_values, axis=1, result_type='expand')

print(all_comb_df)


sub-005
sub-006
sub-007
sub-008
sub-009
sub-010
sub-011
sub-012
sub-013
sub-014
sub-015
sub-016
sub-017
sub-018
sub-019
sub-020
sub-021
sub-022
sub-024
sub-025
sub-026
sub-027
sub-028
sub-029
sub-030
sub-031
sub-032
sub-033
sub-034
sub-035
sub-036
sub-037
sub-038
sub-039
sub-040
sub-041
sub-042
sub-043
sub-045
sub-046
sub-047
sub-048
sub-049
sub-050
sub-051
sub-052
sub-054
sub-056
sub-057
sub-058
sub-059
sub-060
sub-061
sub-062
sub-063
sub-064
       R_Value  Node_Number Condition  Subject  \
0     0.229075            1      cont  sub-005   
1     0.480980            1      cont  sub-005   
2     0.172806            1      cont  sub-005   
3     0.109867            1      cont  sub-005   
4     0.281109            1      cont  sub-005   
...        ...          ...       ...      ...   
4495  0.395086          100      expt  sub-064   
4496  0.027265          100      expt  sub-064   
4497 -0.185836          100      expt  sub-064   
4498  0.146588          100      expt  sub-064   
44

In [10]:

all_comb_df.to_csv(f'../../data/subject_by_node_Viewing1_Viewing2_subs_schaefer_rois.csv')